# Llama 3.2 3B Instruct — Quantisation Sweep & MLX Analysis
Full benchmark across Q2_K → Q8_0 plus MLX backend comparison.
Key metric: **tokens/joule** — throughput per unit of battery drain.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
import glob, json

plt.rcParams.update({
    'figure.facecolor':'#080b10','axes.facecolor':'#0e1219',
    'axes.edgecolor':'#1e2738','axes.labelcolor':'#dde4f0',
    'xtick.color':'#4a5568','ytick.color':'#4a5568',
    'text.color':'#dde4f0','grid.color':'#1e2738',
    'grid.linewidth':0.7,'font.family':'monospace',
    'axes.titlesize':11,'axes.labelsize':9,
})

ACCENT  = '#00e5ff'
ACCENT2 = '#7c9ef8'
PLOTS = Path('../results/plots'); PLOTS.mkdir(parents=True, exist_ok=True)

# Load sweep results (use Phase 1 seed if no sweep run yet)
files = sorted(glob.glob('../results/raw/sweep_*.csv'), reverse=True)
if files:
    df = pd.read_csv(files[0])
    print(f'Loaded: {files[0]}')
else:
    print('No sweep CSV found — using Phase 1 seed data only.')
    df = pd.DataFrame([json.loads('{"quant":"Q4_K_M","size_gib":1.87,"pp_tps":110.43,"pp_std":0.36,"tg_tps":19.22,"tg_std":0.14,"peak_ram_mib":2114.1,"ppl":12.325,"ppl_unc":0.512}')])

# Load MLX results if present
mlx_files = sorted(glob.glob('../results/raw/mlx_benchmark_*.csv'), reverse=True)
mlx = pd.read_csv(mlx_files[0]) if mlx_files else None
if mlx is not None: print(f'MLX results: {mlx_files[0]}')

QUANT_ORDER = ['Q2_K','Q3_K_M','Q4_K_M','Q5_K_M','Q6_K','Q8_0']
df['quant'] = pd.Categorical(df['quant'], categories=QUANT_ORDER, ordered=True)
df = df.sort_values('quant').reset_index(drop=True)
print(df[['quant','size_gib','pp_tps','tg_tps','peak_ram_mib','ppl','tokens_per_joule']].to_string(index=False))

## 1. Efficiency vs Accuracy — The Core Tradeoff

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'Llama 3.2 3B Instruct — Quantisation Sweep', fontweight='bold', y=1.01)

quant_labels = df['quant'].tolist()
x = np.arange(len(quant_labels))
colors = plt.cm.cool(np.linspace(0.2, 0.9, len(x)))

# Left: PPL (accuracy)
ax = axes[0]
if 'ppl' in df.columns and df['ppl'].notna().any():
    bars = ax.bar(x, df['ppl'], color=colors, edgecolor='none', width=0.6)
    if 'ppl_unc' in df.columns:
        ax.errorbar(x, df['ppl'], yerr=df['ppl_unc'].fillna(0),
                    fmt='none', ecolor='#4a5568', capsize=5, lw=1.5)
    ax.set_xticks(x); ax.set_xticklabels(quant_labels, rotation=30, ha='right')
    ax.set_ylabel('Perplexity (lower = better)')
    ax.set_title('PPL — Output Quality', fontweight='bold')
    ax.grid(axis='y', alpha=0.4)
    # Q8_0 as reference line
    q8_ppl = df[df['quant']=='Q8_0']['ppl']
    if len(q8_ppl):
        ax.axhline(q8_ppl.values[0], color=ACCENT, lw=1, ls='--', alpha=0.6,
                   label='Q8_0 baseline')
        ax.legend(fontsize=8, framealpha=0.3)
    for bar, val in zip(bars, df['ppl']):
        ax.text(bar.get_x()+bar.get_width()/2, val+0.1,
                f'{val:.2f}', ha='center', fontsize=8, color='#dde4f0')
else:
    ax.text(0.5, 0.5, 'Run sweep with PPL enabled\npython benchmarks/run_sweep.py',
            ha='center', va='center', transform=ax.transAxes, color='#4a5568', fontsize=9)

# Right: Size vs speed scatter
ax = axes[1]
sc = ax.scatter(df['size_gib'], df['tg_tps'], c=x, cmap='cool',
                s=120, zorder=5, edgecolors='#1e2738', lw=1)
for _, row in df.iterrows():
    ax.annotate(row['quant'], (row['size_gib'], row['tg_tps']),
                xytext=(5, 4), textcoords='offset points', fontsize=8, color='#dde4f0')
ax.set_xlabel('Model size (GiB)')
ax.set_ylabel('Token generation (t/s)')
ax.set_title('Size vs Speed', fontweight='bold')
ax.grid(alpha=0.35)

plt.tight_layout()
plt.savefig(PLOTS/'sweep_tradeoff.png', dpi=150, bbox_inches='tight', facecolor='#080b10')
plt.show()

## 2. Full Metrics Grid

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('All Benchmark Metrics by Quantisation Level', fontweight='bold', y=1.01)

metrics = [
    ('pp_tps',          'Prompt Processing (pp t/s)',  't/s', True),
    ('tg_tps',          'Token Generation (tg t/s)',   't/s', True),
    ('peak_ram_mib',    'Peak RAM',                    'MiB', False),
    ('tokens_per_joule','Tokens per Joule',            'tok/J', True),
]

colors = plt.cm.cool(np.linspace(0.2, 0.9, len(df)))

for ax, (col, title, ylabel, higher_better) in zip(axes.flat, metrics):
    if col not in df.columns or df[col].isna().all():
        ax.text(0.5, 0.5, f'{col}\nnot yet measured',
                ha='center', va='center', transform=ax.transAxes,
                color='#4a5568', fontsize=9)
        ax.set_title(title, fontweight='bold')
        continue
    bars = ax.bar(quant_labels, df[col], color=colors, edgecolor='none', width=0.6)
    ax.set_title(title + (' ↑' if higher_better else ' ↓'), fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.grid(axis='y', alpha=0.4)
    ax.tick_params(axis='x', rotation=30)
    for bar, val in zip(bars, df[col]):
        if pd.notna(val):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
                    f'{val:.1f}', ha='center', fontsize=7.5, color='#dde4f0')

plt.tight_layout()
plt.savefig(PLOTS/'sweep_all_metrics.png', dpi=150, bbox_inches='tight', facecolor='#080b10')
plt.show()

## 3. MLX vs llama.cpp Comparison

In [ ]:
if mlx is None:
    print('No MLX results yet. Run: python mlx/run_mlx_benchmark.py')
else:
    # Compare best llama.cpp quant vs MLX on pp/tg/RAM/tok_per_joule
    # 'Best' = highest tokens/joule, or Q4_K_M if no power data
    if 'tokens_per_joule' in df.columns and df['tokens_per_joule'].notna().any():
        best_row = df.loc[df['tokens_per_joule'].idxmax()]
    else:
        best_row = df[df['quant']=='Q4_K_M'].iloc[0] if 'Q4_K_M' in df['quant'].values else df.iloc[0]

    best_quant = best_row['quant']
    mlx_row    = mlx.iloc[0]

    compare_metrics = ['pp_tps','tg_tps','peak_ram_mib','tokens_per_joule']
    labels_c = ['pp t/s','tg t/s','RAM (MiB)','tok/J']
    higher   = [True, True, False, True]

    llamacpp_vals = [float(best_row.get(m, 0) or 0) for m in compare_metrics]
    mlx_vals      = [float(mlx_row.get(m, 0) or 0) for m in compare_metrics]

    fig, axes = plt.subplots(1, 4, figsize=(14, 4.5))
    fig.suptitle(f'llama.cpp ({best_quant}) vs MLX — {'Llama 3.2 3B Instruct'}',
                 fontweight='bold', y=1.02)

    for ax, metric, label, hb, lv, mv in zip(
            axes, compare_metrics, labels_c, higher, llamacpp_vals, mlx_vals):
        vals = [lv, mv]
        clrs = [ACCENT, ACCENT2]
        bars = ax.bar(['llama.cpp', 'MLX'], vals, color=clrs, edgecolor='none', width=0.5)
        ax.set_title(label + (' ↑' if hb else ' ↓'), fontweight='bold', fontsize=10)
        ax.grid(axis='y', alpha=0.4)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02,
                    f'{val:.1f}', ha='center', fontsize=9, fontweight='bold')
        # Winner annotation
        if lv and mv:
            winner = 'MLX' if (hb and mv>lv) or (not hb and mv<lv) else 'llama.cpp'
            ax.text(0.5, 0.02, f'▲ {winner}', ha='center', transform=ax.transAxes,
                    fontsize=8, color=ACCENT if winner=='llama.cpp' else ACCENT2)

    plt.tight_layout()
    plt.savefig(PLOTS/'mlx_vs_llamacpp.png', dpi=150, bbox_inches='tight', facecolor='#080b10')
    plt.show()

## 4. Interpretation

### Reading the results
- **Q8_0** is the accuracy ceiling at this parameter count — treat its PPL as your baseline
- **Q4_K_M** is typically where the knee of the PPL curve falls — quality holds, size drops significantly
- **Q2_K / Q3_K_M** — expect noticeable PPL degradation; useful only if RAM is the hard constraint
- **tokens/joule** is the edge-deployment metric — it penalises models that are fast but power-hungry

### MLX vs llama.cpp
MLX offloads compute to the M1 GPU and occasionally the ANE, reducing CPU power draw.
Even if raw t/s is similar, expect better tokens/joule under MLX because package power is lower.
This is the key finding that ties the project to the edge deployment brief.

### Recommended configuration
Run `python benchmarks/run_sweep.py` to fill in all cells, then identify:
1. The lowest quant level where PPL stays within 5% of Q8_0 → use this for latency-critical apps
2. The quant level with highest tokens/joule → use this for battery-critical apps
3. Whether MLX beats that quant level on tokens/joule → if yes, MLX is the recommended runtime